In [ ]:
from torch.utils.data import Dataset,DataLoader
from typing import Tuple, List, Dict, Any, Optional
from transformers import AutoModelForCausalLM, AutoTokenizer
import itertools
from utils import load_template,get_r1_prompts,get_r1_ground_truths_with_template,get_device
from dataclasses import dataclass
import torch
from vllm import LLM,SamplingParams
from vllm_utils import init_vllm,load_policy_into_vllm_instance,log_generation,evaluate_vllm
import random
from sft import *
import wandb
import os
from drgrpo_grader import r1_zero_reward_fn
from trainer import SFTConfig,SFTTrainer

INFO 03-04 16:49:58 __init__.py:190] Automatically detected platform cuda.


In [ ]:
from config import SFTConfig
config = TinyConfig()

In [ ]:
device1 = get_device(0)
device2 = get_device(1)

In [ ]:
model_name ='../model/Qwen2.5-Math-1.5B'
model = AutoModelForCausalLM.from_pretrained(
        pretrained_model_name_or_path=model_name,
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
        device_map=device1,# device 1
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)

In [ ]:
vllm = init_vllm(model_name,device=device2,seed=42)

In [ ]:
trainer = SFTTrainer(
    model,
    tokenizer,
    optimizer,
    config,
    vllm)
trainer

In [ ]:
trainer.train()

: 